# 10 — 

In [ ]:
%matplotlib inline
from shared_setup import *

from models.BE_core import BEModel, BEParams, BEState
from models.SC_core import SCModel, SCParams, SCState

from behav_utils.analysis.summary_stats import get_stat_names_expanded, flatten_stats

## Synthetic

In [ ]:
be_params = []
sc_params = []
be_params_array = []
sc_params_array = []
be_names = BEParams.get_param_names()
sc_names = SCParams.get_param_names()

# Default parameter sets (typical values)
for i in range(1000):
    rng = np.random.default_rng(i)
    be_params.append(BEParams.sample_prior(rng=rng))
    sc_params.append(SCParams.sample_prior(rng=rng))
    be_params_array.append(BEParams.sample_prior(rng=rng).to_array())
    sc_params_array.append(SCParams.sample_prior(rng=rng).to_array())
be_params_array = np.array(be_params_array)
sc_params_array = np.array(sc_params_array)

In [ ]:
from behav_utils.data.synthetic import sample_stimuli

In [ ]:
# Generate a uniform stimulus sequence
n_trials = 500
stimuli, categories = sample_stimuli(n_trials=n_trials, rng=np.random.default_rng(42))
no_response = np.zeros(n_trials, dtype=bool)
not_blockstart = np.ones(n_trials, dtype=bool)
not_blockstart[0] = False

In [ ]:
be_stats = []
sc_stats = []

be_um = []
sc_um = []
be_cond_psyc = []
sc_cond_psyc = []

be_stats_without_um = []
sc_stats_without_um = []

for param_idx in range(len(be_params)):
	## Maybe add a pretraining phase to the simulation, where the model is trained on a set of stimuli before the main session?
	# Simulate both models
	be_choices, be_boundaries, be_state, be_trace = BEModel.simulate_session(
		stimuli=stimuli, categories=categories,
		params=be_params[param_idx], initial_state=BEState.initial_uniform(),
		rng=np.random.default_rng(42),
		no_response=no_response, not_blockstart=not_blockstart,
	)

	sc_choices, sc_boundaries, sc_state, sc_trace = SCModel.simulate_session(
		stimuli=stimuli, categories=categories,
		params=sc_params[param_idx], initial_state=SCState.initial_default(),
		rng=np.random.default_rng(42),
		no_response=no_response, not_blockstart=not_blockstart,
	)
	be_stats_dict = fit_summary_stats(be_choices, stimuli, categories, stat_names = list_available_stats(),
                             return_dict = True)
	be_stats_session = flatten_stats(be_stats_dict)
	sc_stats_dict = fit_summary_stats(sc_choices, stimuli, categories, stat_names = list_available_stats(),
							 return_dict = True)
	sc_stats_session = flatten_stats(sc_stats_dict)
	be_stats.append(be_stats_session)
	sc_stats.append(sc_stats_session)

	um_stats_dict = be_stats_dict.pop('update_matrix')
	um_stats = flatten_stats(um_stats_dict)
	be_um.append(um_stats)

	um_stats_dict = sc_stats_dict.pop('update_matrix')
	um_stats = flatten_stats(um_stats_dict)
	sc_um.append(um_stats)

	conditional_psych_dict = be_stats_dict.pop('conditional_psychometric')
	cond_psych_stats = flatten_stats(conditional_psych_dict)
	sc_cond_psyc.append(cond_psych_stats)
 
	conditional_psych_dict = sc_stats_dict.pop('conditional_psychometric')
	cond_psych_stats = flatten_stats(conditional_psych_dict)
	be_cond_psyc.append(cond_psych_stats)
  
	be_stats_without_um.append(flatten_stats(be_stats_dict))
	sc_stats_without_um.append(flatten_stats(sc_stats_dict))
 
be_stats = np.array(be_stats)
sc_stats = np.array(sc_stats)
be_um = np.array(be_um)
sc_um = np.array(sc_um)
be_cond_psyc = np.array(be_cond_psyc)
sc_cond_psyc = np.array(sc_cond_psyc)
be_stats_without_um = np.array(be_stats_without_um)
sc_stats_without_um = np.array(sc_stats_without_um)


be_stat_names = get_stat_names_expanded(be_stats_dict)
um_names = get_stat_names_expanded(um_stats_dict)
cond_psych_names = get_stat_names_expanded(conditional_psych_dict)

### Corr(param, stats)

In [ ]:
# # corr(BE parameters, BE summary statistics) --> 4 x 128 where 4 is the number of parameters and 128 is the number of summary statistics
# be_corr = np.corrcoef(be_params_array, np.array(be_stats))

# # corr(SC parameters, SC summary statistics) --> 4 x 128 where 4 is the number of parameters and 128 is the number of summary statistics
# sc_corr = np.corrcoef(sc_params_array, np.array(sc_stats))

### Corr(UM, stats) and Corr(cond psych, stats)

In [ ]:
um_corr_be = np.corrcoef(be_stats_without_um, np.array(be_um))
cond_psych_corr_be = np.corrcoef(be_stats_without_um, np.array(be_cond_psyc))

## Real Data

find:
- corr(um, stats)
- cor(cond_psyc, stats)
 where stats are all stats except um and cond_psych

In [ ]:
experiment, info = load_data()
print(f"Mode: {info['mode']}")

In [ ]:
stats = []
for animal_id in experiment.animal_ids:
    animal = experiment.get_animal(animal_id)
    sessions = animal.get_sessions()
    real_stats = compute_summary_stats(sessions, stat_names = list_available_stats(), mode = 'per_session')
    for session_stat in real_stats['per_session']:
        stats_current = session_stat['stats']
        if np.isnan(stats_current['accuracy']):
            pass
        else:
            stats.append(stats_current)
            # handle um cond psych and other stats separately if needed
            